# Task 6 — FAR/FRR vs threshold & ROC
Wykres FAR/FRR w funkcji progu. Krzywa ROC. Uzasadnienie wybranego progu.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, cv2
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.metrics import compute_roc, plot_roc, plot_far_frr_vs_threshold, eer
from src.utils import list_images

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()
TEST_DIR = Path('../data/test')

In [ ]:
# Zbierz pary genuine + impostor
scores, labels = [], []
enrolled_users = set(db.get_all_users())

for user_dir in sorted(TEST_DIR.iterdir()):
    if not user_dir.is_dir(): continue
    user_id = user_dir.name
    for img_path in list_images(user_dir):
        img = cv2.imread(str(img_path))
        if img is None: continue
        emb = get_embedding(app, img)
        if emb is None: continue
        if user_id in enrolled_users:
            refs = db.get_user_embeddings(user_id)
            score = max(cosine_similarity(emb, r) for r in refs)
            scores.append(score); labels.append(1)
        else:
            all_refs = [r for embs in db.get_all().values() for r in embs]
            score = max(cosine_similarity(emb, r) for r in all_refs)
            scores.append(score); labels.append(0)

scores = np.array(scores); labels = np.array(labels)

In [ ]:
fpr, tpr, thresholds = compute_roc(scores, labels)
eer_val, eer_thresh = eer(fpr, tpr, thresholds)
print(f'EER: {eer_val:.4f} at threshold={eer_thresh:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_far_frr_vs_threshold(scores, labels, ax=axes[0])
axes[0].axvline(eer_thresh, color='r', linestyle='--', label=f'EER threshold={eer_thresh:.3f}')
axes[0].legend()
plot_roc(fpr, tpr, ax=axes[1], label='Task 6')
plt.tight_layout()
plt.savefig('../results/task6/far_frr_roc.png', dpi=150)
plt.show()